In [ ]:
import sys
from pathlib import Path

# Allow imports from repository root (run from repo root or notebooks/)
_repo = Path.cwd()
if not (_repo / "run_simulation.py").exists():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))



## How to use this notebook

This is the **core segment workflow** notebook.

Run top-to-bottom:
1. Setup
2. Single-run diagnostics (default + balanced)
3. Multi-run default vs balanced comparison plots

For six-case staffing/permit comparisons under `lognormal_180`, use `run_simulation_with_segments_cases.ipynb`.


## 1) Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from run_simulation import run_simulation, run_multiple_simulations, print_statistics
from permit_simulation import Segment, epa_debris_calendar_metrics, usace_debris_calendar_metrics
from visualize_permits import (
    plot_gantt_single_permit,
    plot_total_time_by_segment_quartiles,
    plot_total_time_by_segment,
    plot_average_waiting_and_service_by_step,
)

NUM_PERMITS = 6571
RANDOM_SEED = 42
N_RUNS = 100


## 2) Single-run diagnostics (default mix)


In [ ]:
print(f"Running simulation with {NUM_PERMITS} permits...")
# Scenario 1: Default segment allocation (~90% custom, 2% pre-approved, 8% self-cert, 80% like-for-like)
sim_default = run_simulation(
    num_permits=NUM_PERMITS,
    random_seed=RANDOM_SEED,
    sequential="standard",
)

stats = sim_default.get_statistics()
print_statistics(stats)


In [ ]:
from permit_simulation import epa_debris_calendar_metrics, usace_debris_calendar_metrics

# One value per simulation: earliest debris job start and latest job end (across all permits).

epa = epa_debris_calendar_metrics(sim_default.completed_permits)
usace = usace_debris_calendar_metrics(sim_default.completed_permits)

print("This simulation — EPA / USACE debris (days from t=0)\n")
print("  First start = earliest service start; last end = latest service end.\n")

if epa["first_service_start"] is not None and epa["last_service_end"] is not None:
    print(
        f"  EPA:   first start = {epa['first_service_start']:.2f}, "
        f"last end = {epa['last_service_end']:.2f}"
    )
else:
    print("  EPA:   (no timestamps)")

if usace["first_service_start"] is not None and usace["last_service_end"] is not None:
    print(
        f"  USACE: first start = {usace['first_service_start']:.2f}, "
        f"last end = {usace['last_service_end']:.2f}"
    )
else:
    print("  USACE: (no timestamps)")


In [ ]:
# Optional: Access individual permit data
print(f"\nFirst 5 completed permits:")
for permit in sim_default.completed_permits[:5]:
    total_time = permit.ready_for_construction - permit.created_at if permit.ready_for_construction else None
    print(f"  Permit {permit.permit_id} ({permit.segment.name}): "
          f"{total_time:.2f} days total, "
          f"{permit.public_works_rechecks} re-checks")


In [ ]:
# Gantt chart for one permit in segment 4 (CUSTOM_NON_LIKE)
# Parallel activities (e.g. Public Works, Fire Review, Public Health) appear on separate rows
permit_index = 2345

fig, ax = plot_gantt_single_permit(
    sim_default.completed_permits[permit_index],
    figsize=(16, 8),
)
if fig:
    plt.show()

print(sim_default.completed_permits[permit_index])


In [ ]:
fig, ax = plot_total_time_by_segment_quartiles(sim_default.completed_permits)
plt.show()


## 3) Single-run diagnostics (balanced mix)


In [ ]:
print(f"Running simulation with {NUM_PERMITS} permits...")
# Scenario 2: Balanced segment allocation (50% pre-approved, 25% custom, 25% self-cert, 80% like-for-like)
sim_balanced = run_simulation(
    num_permits=NUM_PERMITS,
    random_seed=RANDOM_SEED,
    sequential="standard",
    pct_pre_approved=0.5,
    pct_custom=0.25,
    pct_self_cert=0.25,
    pct_like_for_like=0.8,
)

stats = sim_balanced.get_statistics()
print_statistics(stats)


## 4) Multi-run comparison (default vs balanced)


In [ ]:
# Run multiple simulations for both segment-mix scenarios to see aggregate behavior

scenario_params_list = [
    {
        "name": "Default",
        "sequential": "standard",
    },
    {
        "name": "Balanced",
        "sequential": "standard",
        "pct_pre_approved": 0.5,
        "pct_custom": 0.25,
        "pct_self_cert": 0.25,
        "pct_like_for_like": 0.8,
    },
]

multi_results = run_multiple_simulations(
    n_runs=N_RUNS,
    num_permits=NUM_PERMITS,
    simulation_duration=None,
    base_seed=42,
    scenario_params_list=scenario_params_list,
    collect_permits=True,
)

all_default_permits: list = []
all_balanced_permits: list = []

for res in multi_results:
    scenario = res["scenario"]
    permits = res.get("permits", [])
    if scenario == "Default":
        all_default_permits.extend(permits)
    elif scenario == "Balanced":
        all_balanced_permits.extend(permits)


def _print_summary(name: str, permits: list) -> None:
    if not permits:
        print(f"{name}: no completed permits across runs")
        return
    total_times = np.array(
        [
            p.ready_for_construction - p.created_at
            for p in permits
            if getattr(p, "ready_for_construction", None) is not None
        ]
    )
    if total_times.size == 0:
        print(f"{name}: no permits with ready_for_construction timestamps")
        return

    print(
        f"{name}: n={len(total_times)}, mean={total_times.mean():.2f}, "
        f"median={np.median(total_times):.2f}"
    )


print(f"Ran {n_runs} runs per segment-mix scenario. Aggregate total-time stats:")
_print_summary("Default", all_default_permits)
_print_summary("Balanced", all_balanced_permits)

In [ ]:
# Average (across runs) of each run's earliest debris start and latest debris end.

import statistics
from collections import defaultdict

by_scenario = defaultdict(
    lambda: {"epa_fs": [], "epa_le": [], "us_fs": [], "us_le": []}
)
all_epa_fs, all_epa_le, all_us_fs, all_us_le = [], [], [], []

for res in multi_results:
    st = res.get("stats") or {}
    if st.get("message"):
        continue
    epa = st.get("debris_removal_epa") or {}
    usc = st.get("debris_removal_usace") or {}
    e_fs = epa.get("first_service_start")
    e_le = epa.get("last_service_end")
    u_fs = usc.get("first_service_start")
    u_le = usc.get("last_service_end")
    if None in (e_fs, e_le, u_fs, u_le):
        continue
    scen = res["scenario"]
    b = by_scenario[scen]
    for lst, v in (
        (b["epa_fs"], e_fs),
        (b["epa_le"], e_le),
        (b["us_fs"], u_fs),
        (b["us_le"], u_le),
    ):
        lst.append(v)
    all_epa_fs.append(e_fs)
    all_epa_le.append(e_le)
    all_us_fs.append(u_fs)
    all_us_le.append(u_le)


def _debris_means_line(label, epa_fs, epa_le, us_fs, us_le):
    n = len(epa_fs)
    if n == 0:
        print(f"{label}: no complete runs")
        return
    print(f"{label} (n = {n} runs)")
    print(
        f"  EPA:   mean of first starts = {statistics.mean(epa_fs):.2f} d, "
        f"mean of last ends = {statistics.mean(epa_le):.2f} d"
    )
    print(
        f"  USACE: mean of first starts = {statistics.mean(us_fs):.2f} d, "
        f"mean of last ends = {statistics.mean(us_le):.2f} d"
    )


print("Debris calendar window — averages across simulations (days)\n")
_debris_means_line("All scenarios", all_epa_fs, all_epa_le, all_us_fs, all_us_le)
print()
for scen in sorted(by_scenario.keys()):
    b = by_scenario[scen]
    _debris_means_line(f"  {scen}", b["epa_fs"], b["epa_le"], b["us_fs"], b["us_le"])
    print()


In [ ]:
# Box-and-whisker: Scenario 1 — Default segment allocation (aggregate across runs)
fig, ax = plot_total_time_by_segment(all_default_permits, figsize=(10, 7))
if ax is not None:
    ax.set_title("Total Time from Disaster to Construction Start by Segment", fontsize=12)
plt.show()


In [ ]:
# Box-and-whisker: Scenario 2 — Balanced segment allocation (50% pre-approved, 25% custom, 25% self-cert, 80% like-for-like, aggregate across runs)
fig, ax = plot_total_time_by_segment(all_balanced_permits, figsize=(10, 6))
if ax is not None:
    ax.set_title("Total Time from Disaster to Construction Start by Segment — Scenario 2: Balanced (50% pre-approved, 25% custom, 25% self-cert)", fontsize=12)
plt.show()


In [ ]:
# Default vs Balanced segment mix — total time by segment (same figure)
# Requires: `all_default_permits`, `all_balanced_permits` from the multi-run cell above.

from matplotlib.patches import Patch
from simulation_plot_helpers import show_boxplot_stats_table

if "all_default_permits" not in dir() or "all_balanced_permits" not in dir():
    raise NameError("Run the multi-run cell first to define all_default_permits and all_balanced_permits.")


def _total_times_by_segment(permits: list) -> dict:
    out: dict = {seg: [] for seg in Segment}
    for p in permits:
        if p.ready_for_construction is not None and p.created_at is not None:
            out[p.segment].append(float(p.ready_for_construction - p.created_at))
    return {s: v for s, v in out.items() if v}


segment_order = [
    Segment.CUSTOM_LIKE,
    Segment.PRE_APPROVED_LIKE,
    Segment.SELF_CERT_LIKE,
    Segment.CUSTOM_NON_LIKE,
    Segment.PRE_APPROVED_NON_LIKE,
    Segment.SELF_CERT_NON_LIKE,
]
label_by_segment = {
    Segment.PRE_APPROVED_LIKE: "Pre-approved like",
    Segment.PRE_APPROVED_NON_LIKE: "Pre-approved non-like",
    Segment.CUSTOM_LIKE: "Custom like",
    Segment.CUSTOM_NON_LIKE: "Custom non-like",
    Segment.SELF_CERT_LIKE: "Self-certified like",
    Segment.SELF_CERT_NON_LIKE: "Self-certified non-like",
}

sd = _total_times_by_segment(all_default_permits)
sb = _total_times_by_segment(all_balanced_permits)
segments = [s for s in segment_order if s in sd or s in sb]
if not segments:
    raise RuntimeError("No segment data in one or both scenarios.")

fig, ax = plt.subplots(figsize=(14, 7))
stats_pairs = []
box_w = 0.32
color_default = "#4C72B0"
color_balanced = "#DD8452"

for i, s in enumerate(segments):
    dd, bd = sd.get(s, []), sb.get(s, [])
    if dd:
        ax.boxplot(
            dd,
            positions=[i - box_w / 2],
            widths=box_w,
            patch_artist=True,
            boxprops=dict(facecolor=color_default, alpha=0.85, edgecolor="black"),
            medianprops=dict(color="darkred", linewidth=2),
            meanprops=dict(marker="o", markeredgecolor="black", markerfacecolor="white", markersize=5),
            showmeans=True,
        )
        stats_pairs.append((f"{label_by_segment[s]} — default mix", dd))
    if bd:
        ax.boxplot(
            bd,
            positions=[i + box_w / 2],
            widths=box_w,
            patch_artist=True,
            boxprops=dict(facecolor=color_balanced, alpha=0.85, edgecolor="black"),
            medianprops=dict(color="darkred", linewidth=2),
            meanprops=dict(marker="o", markeredgecolor="black", markerfacecolor="white", markersize=5),
            showmeans=True,
        )
        stats_pairs.append((f"{label_by_segment[s]} — balanced mix", bd))

ax.set_xticks(range(len(segments)))
ax.set_xticklabels([label_by_segment[s] for s in segments], rotation=45, ha="right")
ax.set_ylabel("Total time (days)", fontsize=12)
ax.set_title(
    "Total time from disaster to construction start by segment\nDefault mix vs Balanced mix (aggregate across runs)",
    fontsize=14,
    fontweight="bold",
)
ax.grid(axis="y", alpha=0.3)
for y_line, label in [(365, "1 yr"), (730, "2 yr")]:
    ax.axhline(y=y_line, color="gray", linestyle="--", linewidth=1, alpha=0.55)
ax.legend(
    handles=[
        Patch(facecolor=color_default, edgecolor="black", label="Default (~2% / 90% / 8%, 80% L4L)"),
        Patch(facecolor=color_balanced, edgecolor="black", label="Balanced (50% / 25% / 25%, 80% L4L)"),
    ],
    loc="upper left",
    fontsize=10,
)
plt.tight_layout()
plt.show()

show_boxplot_stats_table(
    stats_pairs,
    heading="Total time disaster → construction by segment (days; per permit in pooled runs)",
)


In [ ]:
# Visualize the time each permit spends in each stage
from visualize_permits import plot_average_waiting_and_service_by_step


# Visualize aggregate time spent in each process stage across all runs
label_map = {
    "EPA Debris": "Remove debris (EPA)",
    "USACE Debris": "Remove debris (USACE)",
    "Pre-Application Activities": "Pre-application activities",
    "Planning": "Planning review",
    "Special Zoning Review": "Special zoning review",
    "Public Works": "Building & safety review",
    "Agency Referral": "Agency referrals",
    "Fire Review": "Fire review",
    "Applicant Revisions": "Applicant revisions",
}
plot_average_waiting_and_service_by_step(sim_default.completed_permits, label_map=label_map)